In [1]:
from pyprojroot import here
print("Project root:", here())

Project root: C:\Users\hanis\Flight\flight-delay-prediction


### Data obtained so far:

In [2]:
import pandas as pd

# Load train data
df = pd.read_parquet(here("data/processed/reduced.parquet"))
print(df.columns)

Index(['DayOfWeek', 'Reporting_Airline', 'OriginAirportID', 'DestAirportID',
       'CRSDepTime', 'DepDel15', 'CRSArrTime', 'Distance'],
      dtype='object')


# Feature engineering:



## Experiment 0 — Baseline Features

### Feature Set
1. DayOfWeek
2. Reporting_Airline
3. Distance
4. OriginAirportID
5. DestAirportID
   
### Hypothesis

These features capture basic scheduling and operational information available before departure. The goal is to establish a baseline performance level against which all future feature engineering efforts can be measured.

### Result

|Model|PR-AUC|
|---|---|
|Dummy|0.143774|
|Logistic Regression|0.166403|
|Random Forest|0.177326|
|Decision Tree|0.176441|
|XGBoost|0.184916|

### Interpretation

The model achieves a PR-AUC above the no-skill baseline, indicating that even basic operational attributes contain predictive information about flight delays. However, the performance suggests that additional temporal and airport-related factors may be required to capture more complex delay patterns.

---


## Experiment 1 — Time Features

### Added Features
1. dep_hour

### Motivation

The model included departure hour (`dep_hour`). Additional temporal features were introduced to capture broader scheduling patterns:

- `arr_hour`: scheduled arrival hour
- `is_weekend`: distinction between weekday and weekend travel
- `part_of_day`: Morning, Afternoon, Evening, Night

These features were expected to capture differences in passenger demand and operational conditions across different times of the day and week.

### Hypothesis

Flight delays may vary between weekdays and weekends, and between different parts of the day. Scheduled arrival time may also contain additional information about route scheduling and airport congestion.

### Results

|Model|PR-AUC|
|---|---|
|XGBoost|0.224451|
|Logistic Regression|0.199693|
|Random Forest|0.196138|
|Decision Tree|0.182109|
|Dummy|0.143774|

### Interpretation

The additional temporal features produced only a marginal improvement in PR-AUC (+0.003).

This suggests that the majority of the useful temporal information was already captured by `dep_hour`. Since `part_of_day` is derived directly from `dep_hour`, it introduces little new information. Similarly, weekend effects and arrival-hour information appear to contribute limited predictive signal relative to the baseline features.

### Conclusion

The experiment indicates that simple transformations of existing scheduling variables provide limited benefit once departure hour is already included. Consequently, subsequent feature engineering efforts focused on introducing new sources of information, such as airport and route characteristics, rather than creating additional time-based features.

## Experiment 2 — Target encoding features(historical-reliability features)

### Added Features
1. airline_delay_rate
2. origin_delay_rate
3. dest_delay_rate


### Experiment 3 — Historical Reliability Features (Smoothed Target Encoding)

#### Motivation

Airlines and airports exhibit persistent operational characteristics that influence delay likelihood. Rather than representing airlines and airports solely as categorical identifiers, historical delay rates were computed to summarize past operational reliability.

#### Methodology

Three historical reliability features were created:

* `airline_delay_rate`
* `origin_delay_rate`
* `dest_delay_rate`

These features replaced the original high-cardinality categorical variables (`Reporting_Airline`, `OriginAirportID`, and `DestAirportID`) in order to evaluate whether historical operational reliability could serve as a compact substitute for raw airport and airline identifiers.

Smoothed target encoding was used to compute these features. For each category, the historical delay rate was blended with the global training-set delay rate using a smoothing parameter of **m = 100**, reducing the influence of sparse categories while preserving signal from frequently observed categories.

To prevent target leakage:

* Delay-rate mappings were computed exclusively from the training data.
* The same mappings were applied to validation data.
* Previously unseen categories were assigned the global training-set delay rate.

An initial leave-one-out encoding implementation introduced a train-validation distribution mismatch. This issue was detected through abnormal model behavior and was corrected by switching to per-category smoothed target encoding. The final implementation produced consistent train and validation feature distributions while remaining leakage-safe.

As a sanity check, the resulting encoded features retained meaningful variation (e.g., airline delay rates ranged from approximately 0.10 to 0.25 with a standard deviation of approximately 0.036), indicating that the smoothing procedure preserved category-level differences rather than collapsing values toward the global mean.

#### Hypothesis

Historical reliability measures provide a compact representation of airline and airport performance and may capture much of the predictive signal contained within raw categorical identifiers.

#### Results

| Model               | PR-AUC |
| ------------------- | ------ |
| Dummy               | 0.143774 |
| Decision Tree       | 0.182901 |
| Random Forest       | 0.196289  |
| Logistic Regression | 0.200529 |
| XGBoost             | 0.223733  |

The previous best XGBoost model achieved approximately **0.222 PR-AUC** using the original airline and airport identifiers. Replacing those identifiers with historical reliability features resulted in a score of approximately **0.224 PR-AUC**.

#### Interpretation

The observed difference was extremely small and falls within the level of variation expected from normal experimental noise. Consequently, there is insufficient evidence to conclude that historical reliability features materially improved predictive performance.

However, the results indicate that compressing airline and airport identifiers into historical delay-rate statistics preserved most of the predictive signal already present in the original identifiers. The encoded representation achieved essentially the same predictive performance while replacing several high-cardinality categorical variables with only three numerical features.

More broadly, the dominant drivers of flight delays—such as weather conditions, upstream aircraft delays, and real-time air traffic congestion—are not present in the schedule-only dataset. These missing factors likely impose an upper bound on achievable performance regardless of encoding strategy.

#### Conclusion

The hypothesis was supported in terms of representation efficiency, but not in terms of measurable predictive improvement.

Historical reliability features achieved predictive performance comparable to the baseline model despite replacing high-cardinality categorical variables with only three numerical features. This suggests that a substantial portion of the predictive signal contained in airline and airport identifiers can be summarized through historical delay behavior.

The observed increase from **0.222 to 0.224 PR-AUC** was not interpreted as a meaningful performance improvement. However, the encoded representation offers advantages in model simplicity, interpretability, and deployment because it avoids large categorical expansions while retaining comparable predictive performance.

This experiment demonstrates that historical delay-rate encodings can serve as an efficient low-dimensional substitute for raw airline and airport identifiers, achieving comparable predictive performance with no measurable loss of information.
